In [1]:
!pip install pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 71.0 MB/s eta 0:00:00


upload document

In [2]:
from google.colab import files
uploaded = files.upload()
pdf_filename = list(uploaded.keys())[0]

Saving Meta-Learning_in_Neural_Networks_A_Survey.pdf to Meta-Learning_in_Neural_Networks_A_Survey.pdf


pdf -> text

In [5]:
import pymupdf   # PyMuPDF; the legacy import name `fitz` also works

def load_pdf(path: str) -> str:
    """Read a PDF file and return all of its text as one string.

    Each page's text is extracted and the pages are joined together.
    `sort=True` asks PyMuPDF to order text top-left -> bottom-right, which
    helps with the multi-column layouts that are common in papers.
    (If this ever raises an error on your PyMuPDF version, just remove sort=True.)
    """
    doc = pymupdf.open(path)
    pages_text = []
    for page in doc:                       # a document is iterable over its pages
        pages_text.append(page.get_text(sort=True))
    doc.close()
    return "\n".join(pages_text)           # join pages with a newline

# Run the loader on the uploaded file
document_text = load_pdf(pdf_filename)

print(f"Total characters extracted: {len(document_text)}")
print("----- Preview: first 1000 characters -----")
print(document_text[:10000])

Total characters extracted: 171296
----- Preview: first 1000 characters -----
IEEE TRANSACTIONS ON PATTERN ANALYSIS AND MACHINE INTELLIGENCE, VOL. 44, NO. 9, SEPTEMBER 2022                                  5149

  Meta-Learning in Neural Networks: A Survey

             Timothy Hospedales   , Antreas Antoniou, Paul Micaelli, and Amos Storkey



     Abstract—The ﬁeld of meta-learning, or learning-to-learn, has seen a dramatic rise in interest in recent years. Contrary to conventional
      approaches to AI where tasks are solved from scratch using a ﬁxed learning algorithm, meta-learning aims to improve the learning
       algorithm itself, given the experience of multiple learning episodes. This paradigm provides an opportunity to tackle many conventional
      challenges of deep learning, including data and computation bottlenecks, as well as generalization. This survey describes the
      contemporary meta-learning landscape. We ﬁrst discuss deﬁnitions of meta-learning and position 

chunk

In [6]:
def chunk_text(text: str, chunk_size: int = 800, overlap: int = 100) -> list[str]:
    """Split `text` into fixed-size chunks that overlap each other.

    - Every chunk is `chunk_size` characters long (the last one may be shorter).
    - Consecutive chunks share `overlap` characters, so a sentence sitting on a
      boundary still appears whole in at least one chunk.

    The window simply slides forward by (chunk_size - overlap) each step.
    """
    if overlap >= chunk_size:
        raise ValueError("overlap must be smaller than chunk_size")

    chunks = []
    start = 0
    step = chunk_size - overlap            # how far the window moves each time
    while start < len(text):
        chunk = text[start:start + chunk_size]
        chunks.append(chunk)
        start += step
    return chunks

In [7]:
# Parameters chosen for papers: 800-char chunks with 100-char overlap.
CHUNK_SIZE = 800
OVERLAP = 100

chunks = chunk_text(document_text, chunk_size=CHUNK_SIZE, overlap=OVERLAP)

print(f"Number of chunks: {len(chunks)}")
print("\n----- Chunk 0 -----")
print(chunks[0])
print("\n----- Chunk 1 -----")
print(chunks[1])

Number of chunks: 245

----- Chunk 0 -----
IEEE TRANSACTIONS ON PATTERN ANALYSIS AND MACHINE INTELLIGENCE, VOL. 44, NO. 9, SEPTEMBER 2022                                  5149

  Meta-Learning in Neural Networks: A Survey

             Timothy Hospedales   , Antreas Antoniou, Paul Micaelli, and Amos Storkey



     Abstract—The ﬁeld of meta-learning, or learning-to-learn, has seen a dramatic rise in interest in recent years. Contrary to conventional
      approaches to AI where tasks are solved from scratch using a ﬁxed learning algorithm, meta-learning aims to improve the learning
       algorithm itself, given the experience of multiple learning episodes. This paradigm provides an opportunity to tackle many conventional
      challenges of deep learning, including data and computation bottlenecks, as well as generalization. This 

----- Chunk 1 -----
enges of deep learning, including data and computation bottlenecks, as well as generalization. This survey describes the
      contempo

In [8]:
# Sanity check: the last `OVERLAP` chars of chunk 0 should equal the
# first `OVERLAP` chars of chunk 1 (this is the shared overlap region).
# (Assumes the document produced at least two full-length chunks.)
tail_of_chunk0 = chunks[0][-OVERLAP:]
head_of_chunk1 = chunks[1][:OVERLAP]

print("Overlap matches:", tail_of_chunk0 == head_of_chunk1)
print("----- The overlapping text -----")
print(tail_of_chunk0)

Overlap matches: True
----- The overlapping text -----
enges of deep learning, including data and computation bottlenecks, as well as generalization. This 
